# 05 — Comparação de métodos de balanceamento (S4 — issue #15)

Este notebook **não reexecuta a análise**: apenas lê os artefatos congelados de
`src.imbalance` (`comparison.csv`, `metadata.json`, `fold_metrics.json`) e exibe a
figura gerada por `src.sprint_plots`.

Reprodução (não executada neste notebook):

```bash
python -m src.imbalance     # gera reports/sprints-3-4/imbalance/*
python -m src.sprint_plots  # gera reports/sprints-3-4/figures/*.png
```

## Metodologia

- **Baseline comum:** as mesmas features aprovadas do baseline (`NUMERIC_FEATURES` +
  `CATEGORICAL_FEATURES`), **sem histórico do paciente** e sem `sms_received`,
  identificadores ou datas brutas.
- **Validação:** `StratifiedGroupKFold` com 5 folds, somente sobre a partição externa
  de treino (87.720 agendamentos, divisão por paciente, seed 42). O teste de 22.169
  agendamentos permanece reservado e **não foi avaliado**.
- **Sem vazamento:** pré-processamento (escala, codificação ordinal/one-hot) e o
  reamostrador são ajustados **dentro de cada fold**, via `imblearn.pipeline.Pipeline`.
- **Estratégias:** `unbalanced` (sem tratamento), `class_weight` (pesos balanceados na
  regressão logística), `undersampling` (subamostragem aleatória da maioria) e `smote`.
- **SMOTENC, não SMOTE comum:** o SMOTE padrão interpolaria categorias como se fossem
  números. Aqui o `SMOTENC` recebe como discretas — sem interpolação — não só
  `gender`, `neighbourhood` e `weekday`, mas também os indicadores clínicos
  `bolsa_familia`, `hypertension`, `diabetes`, `alcoholism` e a contagem `handicap`
  (ver `smote_discrete_features` nos metadados).
- **Métricas:** limiar 0,5 (precisão, recall, F1) e ranking — top 20% dos scores
  (`precision_at_20`, `recall_at_20`), com captura esperada em empates. AP é
  `average_precision_score`; é a métrica de seleção.
- **Incerteza:** os desvios reportados são o **desvio-padrão amostral entre os 5
  folds**, não intervalos de confiança nem testes de significância.
- **Calibração:** `class_weight` altera a calibração das probabilidades — scores de
  modelos com pesos ou reamostragem **não devem ser lidos como probabilidades
  calibradas**. Nenhuma afirmação causal ou de negócio decorre destas tabelas.

In [ ]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image

OUT = ROOT / 'reports/sprints-3-4/imbalance'
comparacao = pd.read_csv(OUT / 'comparison.csv', index_col=0)
metadata = json.loads((OUT / 'metadata.json').read_text())
folds = pd.read_json(OUT / 'fold_metrics.json')

## Resultados congelados

In [ ]:
{chave: metadata[chave] for chave in
 ('cv', 'n_splits', 'development_rows', 'reserved_test_rows', 'test_evaluated',
  'threshold', 'contact_fraction', 'selection_metric', 'history_included')}

In [ ]:
display(comparacao.round(4))
folds

In [ ]:
display(Image(filename=str(ROOT / 'reports/sprints-3-4/figures/imbalance_comparacao.png')))

## Leitura dos resultados

Em cinco folds, AP média ± desvio-padrão: sem balanceamento 0,3066 ± 0,0053;
class_weight 0,3073 ± 0,0050; undersampling 0,3064 ± 0,0051; SMOTENC 0,3031 ± 0,0046.
A diferença de AP entre pesos de classe e baseline é pequena; não há evidência aqui de
superioridade estatística. No limiar 0,5, o recall médio passa de 2,06% sem
balanceamento para 55,73% com class_weight, mas a precisão cai de 36,59% para 31,31%.
Recall@20% permanece próximo de 34,9% para baseline, class_weight e SMOTENC
(undersampling: 34,53%). Portanto, o salto no recall a 0,5 não significa melhora
equivalente no ranking ou no impacto de negócio. Como candidato para a próxima etapa,
class_weight é simples e evita reamostragem, mas não se escolhe um modelo final nem se
ajusta o limiar nesta tarefa. Scores balanceados não devem ser tratados como
probabilidades calibradas. O teste de 22.169 agendamentos permaneceu reservado; estes
são resultados de validação cruzada, não de teste final.